# Phase 10.5B: Open-Meteo Historical Weather Ingestion & Feature Enrichment

## 🎯 Objective
Integrate open-access, historical hourly meteorological telemetry from the **Open-Meteo Historical Weather API**
(temperature, humidity, surface pressure, wind speed, wind direction, precipitation) to supply critical atmospheric dispersion signals.

### Enriched Feature Pipeline (v2 - 114 Features):
1. **Atmospheric Variables**: `temperature_2m`, `relative_humidity_2m`, `surface_pressure`, `wind_speed_10m`, `precipitation`.
2. **Wind Vectors**: `wind_dir_sin`, `wind_dir_cos`.
3. **Pressure Tendency**: `pressure_diff_1h`, `pressure_diff_24h`.
4. **Dispersion & Stagnation**: `stagnation_index = pm2_5 / (wind_speed + 0.5)`, `thermal_moisture_index`.
5. **Meteorological Lags**: [1h, 3h, 6h, 12h, 24h] for temp, humidity, wind, and pressure.
6. **Rolling Statistics**: [6h, 12h, 24h] mean/std for temp, humidity, and wind.

### Validation Protocol:
Evaluated on **training-only chronological validation** (last 15% of training samples) to keep the held-out test partition untouched.

In [ ]:
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

PROJECT_ROOT = Path("..")
sys.path.insert(0, str(PROJECT_ROOT.resolve()))

DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "data" / "models"

## 1. Feature Schema Comparison: v1 (Pollutants-Only) vs. v2 (Weather-Enriched)

In [ ]:
with open(DATA_DIR / "feature_schema.json", "r") as f:
    schema_v1 = json.load(f)
with open(DATA_DIR / "feature_schema_v2_weather.json", "r") as f:
    schema_v2 = json.load(f)

print(f"v1 (Pollutants-Only) Feature Count:    {len(schema_v1['feature_names'])}")
print(f"v2 (Weather-Enriched) Feature Count:   {len(schema_v2['feature_names'])}")
new_weather_feats = [c for c in schema_v2['feature_names'] if c not in schema_v1['feature_names']]
print(f"New Meteorological Features Added ({len(new_weather_feats)}):", new_weather_feats[:15], "...")

## 2. Weather Feature Correlations with EPA AQI and PM2.5

In [ ]:
df_v2 = pd.read_csv(DATA_DIR / "features_v2_weather.csv")
key_wea_cols = [
    "stagnation_index", "wind_speed_10m", "temperature_2m", "relative_humidity_2m",
    "surface_pressure", "pressure_diff_24h", "thermal_moisture_index", "wind_dir_sin", "wind_dir_cos"
]

corr_aqi = df_v2[key_wea_cols].apply(lambda s: df_v2["epa_aqi"].corr(s))
corr_pm = df_v2[key_wea_cols].apply(lambda s: df_v2["pm2_5"].corr(s))

df_corr = pd.DataFrame({"Correlation with EPA AQI": corr_aqi, "Correlation with PM2.5": corr_pm})
display(df_corr.sort_values("Correlation with EPA AQI", ascending=False))

## 3. Training-Only Chronological Validation: Baseline Ridge Gain

In [ ]:
X_train_v1 = np.load(DATA_DIR / "X_train.npy")
y_train_v1 = np.load(DATA_DIR / "y_train.npy")
X_train_v2 = np.load(DATA_DIR / "X_train_v2.npy")
y_train_v2 = np.load(DATA_DIR / "y_train_v2.npy")

val_pct = 0.15
s1, s2 = int(len(X_train_v1) * (1 - val_pct)), int(len(X_train_v2) * (1 - val_pct))

m1 = Ridge(alpha=1.0).fit(X_train_v1[:s1], y_train_v1[:s1])
p1 = np.clip(m1.predict(X_train_v1[s1:]), 0, 500)

m2 = Ridge(alpha=1.0).fit(X_train_v2[:s2], y_train_v2[:s2])
p2 = np.clip(m2.predict(X_train_v2[s2:]), 0, 500)

df_res = pd.DataFrame({
    "Feature Set": ["v1: Pollutants Only (64 feats)", "v2: Weather Enriched (114 feats)"],
    "Val RMSE": [round(np.sqrt(mean_squared_error(y_train_v1[s1:], p1)), 2), round(np.sqrt(mean_squared_error(y_train_v2[s2:], p2)), 2)],
    "Val MAE": [round(mean_absolute_error(y_train_v1[s1:], p1), 2), round(mean_absolute_error(y_train_v2[s2:], p2), 2)],
    "Val R²": [round(r2_score(y_train_v1[s1:], p1), 4), round(r2_score(y_train_v2[s2:], p2), 4)],
    "h+1 RMSE": [round(np.sqrt(mean_squared_error(y_train_v1[s1:, 0], p1[:, 0])), 2), round(np.sqrt(mean_squared_error(y_train_v2[s2:, 0], p2[:, 0])), 2)],
})
display(df_res)